In [13]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11300" 
print(os.environ.get("OLLAMA_HOST"))

http://localhost:11300


In [ ]:
pip install -r requirements.txt

In [15]:
import Prompts.ver_5_amazonPrompt as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_5_amazonPrompt import Config
import evaluate
importlib.reload(evaluate)

<module 'evaluate' from '/home/aluc31or/LLM_Project/evaluate.py'>

In [16]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [17]:
def evaluate_model(df, results):
    # Get true labels from the dataframe
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [6]:
def run_prompt_eval(dataset_path, prompt_template, model, text_col="text"):
    
    df_local = pd.read_csv(dataset_path)
    reviews = df_local[text_col].tolist()

    prompt = PromptTemplate(
        input_variables=["text"],
        template=prompt_template,
    )

    llm = OllamaLLM(model=model)
    chain = prompt | llm

    predictions = []
    for review in reviews:
        response = chain.invoke({"text": review})
        label = response.strip().lower()

        if "fake" in label:
            cleaned_label = "fake"
        elif "real" in label:
            cleaned_label = "real"
        else:
            cleaned_label = "fake"  # safety fallback
        
        cleaned_label = cleaned_label.strip('.,!?;:')

        #print(f"Raw output: {response} -> Cleaned: {cleaned_label}")
        predictions.append(cleaned_label)

    accuracy, f1, conf_matrix = evaluate_model(df_local, predictions)
    return df_local, predictions, accuracy, f1, conf_matrix

# 1. Amazon Human vs Computer

In [18]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1


## 1.1 Zero Shot Prompting

In [8]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [9]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.64

F1 Score: 0.6069439895185064

Confusion Matrix:
[[140 260]
 [ 28 372]]


In [10]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_prompt_template2,
    model="llama3:8b"
 )

In [11]:
print("=" * 50)
print("zero_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.73

F1 Score: 0.7299729972997301

Confusion Matrix:
[[288 112]
 [104 296]]


## 1.2 One-Shot Prompting

In [12]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [13]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.68

F1 Score: 0.6795493662963543

Confusion Matrix:
[[257 143]
 [113 287]]


In [14]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_prompt_template2,
    model="llama3:8b"
 )

In [15]:
print("=" * 50)
print("one_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.66875

F1 Score: 0.6686873612042277

Confusion Matrix:
[[273 127]
 [138 262]]


## 1.3 Few-Shot Prompting

In [ ]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [17]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.72625

F1 Score: 0.7161369347649624

Confusion Matrix:
[[366  34]
 [185 215]]


In [19]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_prompt_template2,
    model="llama3:8b"
 )

In [20]:
print("=" * 50)
print("few_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.715

F1 Score: 0.7149982187388672

Confusion Matrix:
[[287 113]
 [115 285]]


# 2. Hotel Human Real vs Human Fake

In [21]:
import Prompts.ver_4_hotelPrompt as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_4_hotelPrompt import Config
print(Config.few_shot_prompt_template)


    You are an expert at detecting real and fake reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Example 1:
    Review: "As a former Chicagoan, I'm appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There's an Experience Designer who is supposed to be like a 'personal concierge,' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted. Not only that, but I couldn't understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room's cleanliness. I had to ask for a maid to c

In [22]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,truthful,conrad,Hotel,We stayed for a one night getaway with family ...,0,TripAdvisor
1,truthful,hyatt,Hotel,Triple A rate with upgrade to view room was le...,0,TripAdvisor
2,truthful,hyatt,Hotel,This comes a little late as I'm finally catchi...,0,TripAdvisor
3,truthful,omni,Hotel,The Omni Chicago really delivers on all fronts...,0,TripAdvisor
4,truthful,hyatt,Hotel,I asked for a high floor away from the elevato...,0,TripAdvisor


## 2.1 Zero Shot Prompting

In [23]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [24]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5131909547738693

F1 Score: 0.36901096124823113

Confusion Matrix:
[[ 28 768]
 [  7 789]]


In [25]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_prompt_template2,
    model="llama3:8b"
 )

In [26]:
print("=" * 50)
print("zero_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.5885678391959799

F1 Score: 0.5627281476247487

Confusion Matrix:
[[275 521]
 [134 662]]


## 2.2 One-Shot Prompting

In [27]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b")

In [28]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.6180904522613065

F1 Score: 0.5952734175733198

Confusion Matrix:
[[303 493]
 [115 681]]


In [29]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_prompt_template2,
    model="llama3:8b")

In [30]:
print("=" * 50)
print("one_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.6268844221105527

F1 Score: 0.6206555714588293

Confusion Matrix:
[[397 399]
 [195 601]]


## 2.3 Few-Shot Prompting

In [31]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [32]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.6118090452261307

F1 Score: 0.5886167303623546

Confusion Matrix:
[[298 498]
 [120 676]]


In [33]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.few_shot_prompt_template2,
    model="llama3:8b"
 )

In [34]:
print("=" * 50)
print("few_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.6413316582914573

F1 Score: 0.6323374707730712

Confusion Matrix:
[[386 410]
 [161 635]]


# 3. Human Real vs CG

In [35]:
import Prompts.ver_4_hotelPrompt as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_4_hotelPrompt import Config
print(Config.few_shot_prompt_template)


    You are an expert at detecting real and fake reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "real" or "fake". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording and lacks details or specificity.

    Here are some examples:

    Example 1:
    Review: "As a former Chicagoan, I'm appalled at the Amalfi Hotel Chicago. First of all, I was expecting luxury and hospitality, neither of which I received. There's an Experience Designer who is supposed to be like a 'personal concierge,' but my experience with my ED was terrible. I felt like he was trying to pressure me into staying more days than I wanted. Not only that, but I couldn't understand what he was saying most of the time because he was talking so fast. When I finally got to my room, I was disappointed with the quality of the furniture and the room's cleanliness. I had to ask for a maid to c

In [36]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web
1,real,palmer,Hotel,Here are my experiences: - Had three rooms res...,0,Web
2,real,omni,Hotel,My wife and I travelled to Chicago and really ...,0,TripAdvisor
3,real,knickerbocker,Hotel,The location of this hotel is excellent. The h...,0,Web
4,real,fairmont,Hotel,We recently completed our second stay at the F...,0,TripAdvisor


## 3.1 Zero Shot Prompting

In [37]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [38]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.49685929648241206

F1 Score: 0.33413642665246374

Confusion Matrix:
[[  2 794]
 [  7 789]]


In [39]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_prompt_template2,
    model="llama3:8b"
 )

In [40]:
print("=" * 50)
print("zero_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.6155778894472361

F1 Score: 0.596287333841615

Confusion Matrix:
[[316 480]
 [132 664]]


## 3.2 One-Shot Prompting

In [41]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [42]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.7701005025125628

F1 Score: 0.7688734030197445

Confusion Matrix:
[[555 241]
 [125 671]]


In [43]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_prompt_template2,
    model="llama3:8b"
 )

In [44]:
print("=" * 50)
print("one_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.7550251256281407

F1 Score: 0.7549133386070704

Confusion Matrix:
[[584 212]
 [178 618]]


## 3.3 Few-Shot Prompting

In [45]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [46]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.835427135678392

F1 Score: 0.8353606163950992

Confusion Matrix:
[[649 147]
 [115 681]]


In [47]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_prompt_template2,
    model="llama3:8b"
 )

In [48]:
print("=" * 50)
print("few_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.7870603015075377

F1 Score: 0.7870077775265122

Confusion Matrix:
[[614 182]
 [157 639]]


# 4. Human Real vs MixFake

In [49]:
from Prompts.ver_4_hotelPrompt import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews.

    Task: Classify the following review of a hotel that is listed on a website as either "fake" or "real". Analyze the language and structure of the review,
	It should not be overly generic or templeted language,should not have sales-like wording.

    Review: "{text}"

    Classify this review as either "fake" or "real" (respond with only one word):


In [50]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,sofitel,Hotel,Just got back from a 10 day visit to Chicago. ...,0,TripAdvisor
1,real,palmer,Hotel,Just returned from a week in Chicago with the ...,0,TripAdvisor
2,real,hardrock,Hotel,Not worth the price. We almost stayed at a hot...,0,Web
3,real,homewood,Hotel,WARNING: FOOD POISONING ALERT!!! The room was ...,0,Web
4,real,ambassador,Hotel,Upon check in - the lobby was BUSY! I was tire...,0,TripAdvisor


## 4.1 Zero Shot Prompting

In [51]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [52]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5069095477386935

F1 Score: 0.3568431524216678

Confusion Matrix:
[[ 19 777]
 [  8 788]]


In [53]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.zero_shot_prompt_template2,
    model="llama3:8b"
 )

In [54]:
print("=" * 50)
print("zero_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.6136934673366834

F1 Score: 0.5920117511016657

Confusion Matrix:
[[305 491]
 [124 672]]


## 4.2 One-Shot Prompting

In [55]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [56]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.717964824120603

F1 Score: 0.7127253451135522

Confusion Matrix:
[[464 332]
 [117 679]]


In [57]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.one_shot_prompt_template2,
    model="llama3:8b"
 )

In [58]:
print("=" * 50)
print("one_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.6947236180904522

F1 Score: 0.6932591806733683

Confusion Matrix:
[[498 298]
 [188 608]]


## 4.3 Few-Shot Prompting

In [59]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [60]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.7267587939698492

F1 Score: 0.722059788530732

Confusion Matrix:
[[475 321]
 [114 682]]


In [61]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.few_shot_prompt_template2,
    model="llama3:8b"
 )

In [62]:
print("=" * 50)
print("few_shot_prompt_template2 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template2 Results
Model: llama3:8b
Accuracy: 0.7198492462311558

F1 Score: 0.718139263098391

Confusion Matrix:
[[511 285]
 [161 635]]
